In [ ]:
# Check that our research environment is ready

import sys
import pandas as pd
import numpy as np
import sklearn
import matplotlib

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

print("\nResearch environment is ready!")

In [ ]:
# Place the official UNSW-NB15 training and testing CSVs in the notebook working directory.
# For Colab, upload them using the Files panel or mount Drive separately.
from pathlib import Path
for filename in ('UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv'):
    if not Path(filename).exists():
        raise FileNotFoundError(f'Missing {filename}; see README.md')


In [ ]:
import pandas as pd

train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df = pd.read_csv("UNSW_NB15_testing-set.csv")

print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

In [ ]:
# Separate input features and labels

columns_to_remove = ["id", "attack_cat", "label"]

X_train_full = train_df.drop(columns=columns_to_remove)
y_train_full = train_df["label"]

X_test = test_df.drop(columns=columns_to_remove)
y_test = test_df["label"]

print("Training features:", X_train_full.shape)
print("Training labels:", y_train_full.shape)

print("Testing features:", X_test.shape)
print("Testing labels:", y_test.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.30,
    random_state=42,
    stratify=y_train_full
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Official test:", X_test.shape)


In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine both types
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Learn transformations from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformations to validation data
X_val_processed = preprocessor.transform(X_val)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
import time

# Create our first machine learning model
model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

# Measure training time
start_time = time.perf_counter()

model.fit(X_train_processed, y_train)

training_time = time.perf_counter() - start_time

print("Model trained successfully!")
print(f"Training time: {training_time:.2f} seconds")


In [ ]:
# Predict on validation data, NOT the official test set

start_time = time.perf_counter()

y_pred = model.predict(X_val_processed)

prediction_time = time.perf_counter() - start_time

print("Predictions completed!")
print("Number of predictions:", len(y_pred))
print(f"Total prediction time: {prediction_time:.4f} seconds")
print(f"Average latency: {prediction_time / len(y_pred) * 1000:.6f} ms/record")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1-score:", f1_score(y_val, y_pred))
print("Macro-F1:", f1_score(y_val, y_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred,
    target_names=["Normal", "Attack"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred,
    display_labels=["Normal", "Attack"],
    cmap="Blues",
    values_format="d"
)

plt.title("Logistic Regression - Validation Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

# Predicted probabilities for the attack class
attack_probabilities = model.predict_proba(
    X_val_processed
)[:, 1]

results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:

    predictions = (
        attack_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "False Positive Rate": fp / (fp + tn),
        "Missed Attacks": fn
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start = time.perf_counter()

rf_model.fit(X_train_processed, y_train)

rf_training_time = time.perf_counter() - start

print("Random Forest trained successfully!")
print(f"Training time: {rf_training_time:.2f} seconds")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

start = time.perf_counter()

rf_pred = rf_model.predict(X_val_processed)

rf_prediction_time = time.perf_counter() - start

tn, fp, fn, tp = confusion_matrix(
    y_val, rf_pred, labels=[0, 1]
).ravel()

print("Accuracy:", round(accuracy_score(y_val, rf_pred), 4))
print("Precision:", round(precision_score(y_val, rf_pred), 4))
print("Recall:", round(recall_score(y_val, rf_pred), 4))
print("F1-score:", round(f1_score(y_val, rf_pred), 4))
print("Macro-F1:", round(f1_score(y_val, rf_pred, average="macro"), 4))
print("False-positive rate:", round(fp / (fp + tn), 4))

print("\nTraining time:", round(rf_training_time, 2), "seconds")
print("Prediction time:", round(rf_prediction_time, 4), "seconds")
print("Average prediction latency:",
      round(rf_prediction_time / len(rf_pred) * 1000, 6),
      "ms/record")

print("\nClassification report:")
print(classification_report(y_val, rf_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, rf_pred))


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

def calculate_metrics(name, y_true, predictions):
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions),
        "Macro-F1": f1_score(
            y_true, predictions, average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": int(fn)
    }

comparison = pd.DataFrame([
    calculate_metrics(
        "Logistic Regression (0.50)", y_val, y_pred
    ),
    calculate_metrics(
        "Random Forest", y_val, rf_pred
    )
])

display(comparison.round(4))


In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Logistic Regression": model,
    "Random Forest": rf_model
}

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
rows = []

for model_name, classifier in models.items():

    probabilities = classifier.predict_proba(
        X_val_processed
    )[:, 1]

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        rows.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_val, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_val, predictions, zero_division=0
            ),
            "FPR": fp / (fp + tn),
            "Missed Attacks": fn
        })

model_threshold_comparison = pd.DataFrame(rows)

display(model_threshold_comparison.round(4))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for model_name in models:

    subset = model_threshold_comparison[
        model_threshold_comparison["Model"] == model_name
    ]

    plt.plot(
        subset["FPR"] * 100,
        subset["Recall"] * 100,
        marker="o",
        label=model_name
    )

plt.xlabel("False Positive Rate (%)")
plt.ylabel("Attack Recall (%)")
plt.title("Detection Trade-off: Logistic Regression vs Random Forest")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd

# Convert the processed data to dense format if needed.
# This is manageable for our current dataset, but may use more RAM.
if hasattr(X_train_processed, "toarray"):
    X_train_dense = X_train_processed.toarray()
    X_val_dense = X_val_processed.toarray()
else:
    X_train_dense = X_train_processed
    X_val_dense = X_val_processed

# Select 20 encoded features using training data only.
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(
        X, y, random_state=42
    ),
    k=20
)

X_train_selected = selector.fit_transform(
    X_train_dense, y_train
)

X_val_selected = selector.transform(X_val_dense)

print("Original shape:", X_train_dense.shape)
print("Selected training shape:", X_train_selected.shape)
print("Selected validation shape:", X_val_selected.shape)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

rf_selected = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start = time.perf_counter()

rf_selected.fit(X_train_selected, y_train)

selected_training_time = time.perf_counter() - start

start = time.perf_counter()

selected_predictions = rf_selected.predict(X_val_selected)

selected_prediction_time = time.perf_counter() - start

print("Training time:", round(selected_training_time, 2), "seconds")
print("Prediction time:", round(selected_prediction_time, 4), "seconds")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_configuration(name, predictions, prediction_time):
    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Configuration": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Recall": recall_score(y_val, predictions),
        "Macro-F1": f1_score(
            y_val, predictions, average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": fn,
        "Prediction Time (s)": prediction_time
    }

feature_comparison = pd.DataFrame([
    evaluate_configuration(
        "Random Forest - 192 encoded features",
        rf_pred,
        rf_prediction_time
    ),
    evaluate_configuration(
        "Random Forest - 20 encoded features",
        selected_predictions,
        selected_prediction_time
    )
])

display(feature_comparison.round(4))


In [ ]:
import numpy as np
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Reuse mutual-information scores from Experiment 3
scores = selector.scores_

# Rank features from most to least informative
ranked_features = np.argsort(
    np.nan_to_num(scores, nan=-np.inf)
)[::-1]

feature_counts = [20, 40, 80, 192]

experiment_results = []

for k in feature_counts:

    print(f"Training model with {k} features...")

    selected_indices = ranked_features[:k]

    X_train_k = X_train_dense[:, selected_indices]
    X_val_k = X_val_dense[:, selected_indices]

    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    start = time.perf_counter()
    rf.fit(X_train_k, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = rf.predict(X_val_k)
    pred_time = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    experiment_results.append({
        "Features": k,
        "Accuracy": accuracy_score(y_val, predictions),
        "Recall": recall_score(y_val, predictions),
        "Macro-F1": f1_score(
            y_val, predictions, average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": fn,
        "Training Time (s)": train_time,
        "Prediction Time (s)": pred_time
    })

results_df = pd.DataFrame(experiment_results)

display(results_df.round(4))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    results_df["Features"],
    results_df["Macro-F1"],
    marker="o",
    label="Macro-F1"
)

plt.plot(
    results_df["Features"],
    results_df["FPR"],
    marker="s",
    label="False-positive rate"
)

plt.xlabel("Number of Selected Features")
plt.ylabel("Metric Value")
plt.title("Effect of Feature Count on Intrusion Detection")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
results_df.to_csv(
    "UNSW_NB15_feature_count_experiment.csv",
    index=False
)

print("Experimental results saved successfully!")


In [ ]:

import pandas as pd
import numpy as np

# Retrieve feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Retrieve mutual-information scores
mi_scores = selector.scores_

# Build a ranked feature table
feature_ranking = pd.DataFrame({
    "Feature": feature_names,
    "MI_Score": mi_scores
})

feature_ranking = feature_ranking.sort_values(
    by="MI_Score",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

print("Top 40 selected encoded features:")
display(feature_ranking.head(40))


In [ ]:
top_40 = feature_ranking.head(40).copy()

top_40["Feature_Type"] = np.where(
    top_40["Feature"].str.startswith("numeric__"),
    "Numerical",
    "Categorical indicator"
)

print("Selected feature types:")
print(top_40["Feature_Type"].value_counts())

print("\nSelected feature names:")
print(top_40["Feature"].tolist())


In [ ]:
feature_ranking.to_csv(
    "UNSW_NB15_mutual_information_feature_ranking.csv",
    index=False
)

print("Feature ranking saved successfully!")


In [ ]:
# Convert the top 40 encoded feature names
# into their original UNSW-NB15 variable names.

original_features = []

for feature in top_40["Feature"]:

    if feature.startswith("numeric__"):
        original_name = feature.replace("numeric__", "", 1)

    elif feature.startswith("categorical__proto_"):
        original_name = "proto"

    elif feature.startswith("categorical__service_"):
        original_name = "service"

    elif feature.startswith("categorical__state_"):
        original_name = "state"

    else:
        raise ValueError(f"Unknown feature name: {feature}")

    original_features.append(original_name)

unique_original_features = list(
    dict.fromkeys(original_features)
)

print("Selected encoded columns:", len(top_40))
print("Distinct original variables:", len(unique_original_features))

print("\nOriginal network variables:")
print(unique_original_features)


In [ ]:
all_original_features = X_train.columns.tolist()

excluded_features = [
    feature for feature in all_original_features
    if feature not in unique_original_features
]

print("Total original input features:", len(all_original_features))
print("Selected original features:", len(unique_original_features))
print("Excluded original features:", len(excluded_features))

print("\nExcluded features:")
print(excluded_features)


In [ ]:
original_feature_table = pd.DataFrame({
    "Original_Feature": unique_original_features
})

original_feature_table.to_csv(
    "UNSW_NB15_selected_original_features.csv",
    index=False
)

print("Original feature list saved successfully!")


In [ ]:
import pandas as pd
import numpy as np

# Start with the encoded feature names and MI scores
encoded_names = preprocessor.get_feature_names_out()
mi_values = selector.scores_

def original_name(encoded_name):
    if encoded_name.startswith("numeric__"):
        return encoded_name.replace("numeric__", "", 1)

    for name in ["proto", "service", "state"]:
        if encoded_name.startswith(f"categorical__{name}_"):
            return name

    raise ValueError(f"Unrecognized feature: {encoded_name}")

ranking = pd.DataFrame({
    "Encoded": encoded_names,
    "Original": [original_name(n) for n in encoded_names],
    "MI": mi_values
})

# Aggregate encoded-column scores into original-variable scores
# Maximum MI is a simple exploratory heuristic.
original_ranking = (
    ranking.groupby("Original", as_index=False)["MI"]
    .max()
    .sort_values("MI", ascending=False)
    .reset_index(drop=True)
)

display(original_ranking.head(20))
print("Original variables ranked:", len(original_ranking))


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score,
    f1_score, confusion_matrix
)
import time

ranked_original = original_ranking["Original"].tolist()

original_results = []

for k in [10, 20, 30, 42]:

    # Select complete original network variables
    selected = ranked_original[:k]

    numeric_cols = [
        c for c in selected
        if c in numerical_features
    ]

    categorical_cols = [
        c for c in selected
        if c in categorical_features
    ]

    transformers = []

    if numeric_cols:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_cols
        ))

    if categorical_cols:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_cols
        ))

    preprocessing = ColumnTransformer(transformers)

    experiment_model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    start = time.perf_counter()
    experiment_model.fit(X_train[selected], y_train)
    train_seconds = time.perf_counter() - start

    start = time.perf_counter()
    predictions = experiment_model.predict(X_val[selected])
    predict_seconds = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    original_results.append({
        "Original Features": k,
        "Encoded Features": experiment_model.named_steps[
            "preprocessing"
        ].transform(X_train[selected].iloc[:1]).shape[1],
        "Accuracy": accuracy_score(y_val, predictions),
        "Recall": recall_score(y_val, predictions),
        "Macro-F1": f1_score(
            y_val, predictions, average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": fn,
        "Training Time (s)": train_seconds,
        "Pipeline Prediction Time (s)": predict_seconds
    })

    print(f"Completed {k} original features")

original_results_df = pd.DataFrame(original_results)
display(original_results_df.round(4))


In [ ]:
original_results_df.to_csv(
    "UNSW_NB15_original_feature_experiment.csv",
    index=False
)

original_ranking.to_csv(
    "UNSW_NB15_original_feature_ranking.csv",
    index=False
)

print("Both research files saved!")


In [ ]:
# Check duplicate network observations
# Exclude the record identifier, but retain both labels.

columns_without_id = [
    col for col in train_df.columns
    if col != "id"
]

train_duplicates = train_df.duplicated(
    subset=columns_without_id
).sum()

test_duplicates = test_df.duplicated(
    subset=columns_without_id
).sum()

print("Training duplicates without ID:", train_duplicates)
print("Testing duplicates without ID:", test_duplicates)


In [ ]:
# Check whether the same network features
# sometimes have different binary labels.

feature_columns = X_train_full.columns.tolist()

conflicting_labels = (
    train_df.groupby(
        feature_columns,
        dropna=False
    )["label"]
    .nunique()
)

print(
    "Feature combinations with conflicting labels:",
    (conflicting_labels > 1).sum()
)


In [ ]:
# Identify identical feature rows shared
# between the training and validation partitions.

train_hashes = pd.util.hash_pandas_object(
    X_train, index=False
)

val_hashes = pd.util.hash_pandas_object(
    X_val, index=False
)

overlap_count = val_hashes.isin(
    set(train_hashes)
).sum()

print(
    "Validation rows with identical features in training:",
    overlap_count
)

print(
    "Validation overlap percentage:",
    round(overlap_count / len(X_val) * 100, 4),
    "%"
)


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Use the official training file only.
# Exclude ID, attack category and target label.
X_all = train_df.drop(
    columns=["id", "attack_cat", "label"]
)

y_all = train_df["label"]

# Identical feature vectors receive the same group ID.
groups = pd.util.hash_pandas_object(
    X_all,
    index=False
)

print("Total records:", len(X_all))
print("Unique feature-vector groups:", groups.nunique())


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(X_all, y_all, groups=groups)
)

X_train_grouped = X_all.iloc[train_indices].copy()
X_val_grouped = X_all.iloc[val_indices].copy()

y_train_grouped = y_all.iloc[train_indices].copy()
y_val_grouped = y_all.iloc[val_indices].copy()

print("Grouped training records:", len(X_train_grouped))
print("Grouped validation records:", len(X_val_grouped))

print("\nTraining attack proportion:")
print(y_train_grouped.mean())

print("\nValidation attack proportion:")
print(y_val_grouped.mean())


In [ ]:
train_group_hashes = pd.util.hash_pandas_object(
    X_train_grouped, index=False
)

val_group_hashes = pd.util.hash_pandas_object(
    X_val_grouped, index=False
)

overlap = val_group_hashes.isin(
    set(train_group_hashes)
).sum()

print("Identical validation features in training:", overlap)

if overlap == 0:
    print("SUCCESS: No exact feature-vector overlap!")
else:
    print("WARNING: Overlap detected.")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Identify column types using grouped training data
categorical_cols = X_train_grouped.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X_train_grouped.select_dtypes(
    include="number"
).columns.tolist()

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
grouped_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import pandas as pd
import time

grouped_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

grouped_results = []

for name, classifier in grouped_models.items():

    # Each pipeline fits preprocessing on grouped training only
    pipeline = Pipeline([
        ("preprocessing", grouped_preprocessor),
        ("classifier", classifier)
    ])

    start = time.perf_counter()
    pipeline.fit(X_train_grouped, y_train_grouped)
    train_seconds = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_val_grouped)
    predict_seconds = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(
        y_val_grouped, predictions, labels=[0, 1]
    ).ravel()

    grouped_results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_val_grouped, predictions
        ),
        "Precision": precision_score(
            y_val_grouped, predictions
        ),
        "Recall": recall_score(
            y_val_grouped, predictions
        ),
        "F1": f1_score(
            y_val_grouped, predictions
        ),
        "Macro-F1": f1_score(
            y_val_grouped, predictions,
            average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": fn,
        "Training Time (s)": train_seconds,
        "Pipeline Prediction Time (s)": predict_seconds
    })

    print(f"Completed: {name}")

grouped_results_df = pd.DataFrame(grouped_results)

display(grouped_results_df.round(4))


In [ ]:

previous_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Validation": "Random split",
        "Accuracy": 0.9353,
        "Recall": 0.9865,
        "Macro-F1": 0.9225,
        "FPR": 0.1736
    },
    {
        "Model": "Random Forest",
        "Validation": "Random split",
        "Accuracy": 0.9565,
        "Recall": 0.9805,
        "Macro-F1": 0.9492,
        "FPR": 0.0946
    }
])

new_results = grouped_results_df[
    ["Model", "Accuracy", "Recall", "Macro-F1", "FPR"]
].copy()

new_results["Validation"] = "Duplicate-aware split"

comparison_split = pd.concat([
    previous_results,
    new_results
], ignore_index=True)

display(
    comparison_split[
        ["Model", "Validation", "Accuracy",
         "Recall", "Macro-F1", "FPR"]
    ].round(4)
)


In [ ]:

grouped_results_df.to_csv(
    "UNSW_NB15_duplicate_aware_baselines.csv",
    index=False
)

comparison_split.to_csv(
    "UNSW_NB15_validation_strategy_comparison.csv",
    index=False
)

print("Research results saved!")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Step 1: Fit preprocessing on grouped training data ONLY
X_grouped_processed = grouped_preprocessor.fit_transform(
    X_train_grouped
)

# Step 2: Convert sparse matrix to dense NumPy array
if hasattr(X_grouped_processed, "toarray"):
    X_grouped_processed = X_grouped_processed.toarray()

print("Processed shape:", X_grouped_processed.shape)
print("Matrix type:", type(X_grouped_processed))

# Step 3: Get encoded feature names
grouped_feature_names = (
    grouped_preprocessor.get_feature_names_out()
)

# Step 4: Identify categorical indicators
discrete_mask = np.array([
    name.startswith("categorical__")
    for name in grouped_feature_names
])

print("Continuous features:", (~discrete_mask).sum())
print("Discrete features:", discrete_mask.sum())

# Step 5: Calculate mutual information
# Using ONLY the grouped training partition
grouped_mi_scores = mutual_info_classif(
    X_grouped_processed,
    y_train_grouped,
    discrete_features=discrete_mask,
    random_state=42
)

# Step 6: Create the feature ranking
grouped_ranking = pd.DataFrame({
    "Encoded": grouped_feature_names,
    "MI": grouped_mi_scores
})

print("\nTop 20 encoded features:")
display(
    grouped_ranking.sort_values(
        "MI", ascending=False
    ).head(20)
)


In [ ]:

def get_original_feature(encoded_name):

    if encoded_name.startswith("numeric__"):
        return encoded_name.replace("numeric__", "", 1)

    for feature in ["proto", "service", "state"]:
        if encoded_name.startswith(
            f"categorical__{feature}_"
        ):
            return feature

    raise ValueError(
        f"Unknown encoded feature: {encoded_name}"
    )

grouped_ranking["Original"] = (
    grouped_ranking["Encoded"].apply(
        get_original_feature
    )
)

grouped_original_ranking = (
    grouped_ranking.groupby(
        "Original", as_index=False
    )["MI"]
    .max()
    .sort_values("MI", ascending=False)
    .reset_index(drop=True)
)

print("Number of original features:",
      len(grouped_original_ranking))

display(grouped_original_ranking.head(20))


In [ ]:

grouped_original_ranking.to_csv(
    "UNSW_NB15_grouped_original_feature_ranking.csv",
    index=False
)

print("Grouped feature ranking saved!")


In [ ]:

import pandas as pd
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix
)

ranked_features = grouped_original_ranking[
    "Original"
].tolist()

feature_counts = [10, 20, 30, 42]

grouped_feature_results = []

for k in feature_counts:

    selected = ranked_features[:k]

    numeric_selected = [
        c for c in selected
        if c in numeric_cols
    ]

    categorical_selected = [
        c for c in selected
        if c in categorical_cols
    ]

    transformers = []

    if numeric_selected:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_selected
        ))

    if categorical_selected:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_selected
        ))

    preprocessing = ColumnTransformer(transformers)

    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    start = time.perf_counter()

    pipeline.fit(
        X_train_grouped[selected],
        y_train_grouped
    )

    training_time = time.perf_counter() - start

    start = time.perf_counter()

    predictions = pipeline.predict(
        X_val_grouped[selected]
    )

    prediction_time = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(
        y_val_grouped,
        predictions,
        labels=[0, 1]
    ).ravel()

    grouped_feature_results.append({
        "Original Features": k,
        "Accuracy": accuracy_score(
            y_val_grouped, predictions
        ),
        "Recall": recall_score(
            y_val_grouped, predictions
        ),
        "Macro-F1": f1_score(
            y_val_grouped, predictions,
            average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": fn,
        "Training Time (s)": training_time,
        "Pipeline Prediction Time (s)": prediction_time
    })

    print(f"Completed: {k} original features")

grouped_feature_results_df = pd.DataFrame(
    grouped_feature_results
)

display(grouped_feature_results_df.round(4))


In [ ]:

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    grouped_feature_results_df["Original Features"],
    grouped_feature_results_df["Macro-F1"] * 100,
    marker="o",
    label="Macro-F1 (%)"
)

ax1.plot(
    grouped_feature_results_df["Original Features"],
    grouped_feature_results_df["FPR"] * 100,
    marker="s",
    label="False-positive rate (%)"
)

ax1.set_xlabel("Number of Original Network Features")
ax1.set_ylabel("Percentage (%)")
ax1.set_title(
    "Duplicate-Aware Feature Selection Performance"
)

ax1.legend()
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

ranked_features = grouped_original_ranking[
    "Original"
].tolist()

benchmark_results = []

for k in [30, 42]:

    selected = ranked_features[:k]

    numeric_selected = [
        c for c in selected if c in numeric_cols
    ]

    categorical_selected = [
        c for c in selected if c in categorical_cols
    ]

    transformers = []

    if numeric_selected:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_selected
        ))

    if categorical_selected:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_selected
        ))

    preprocessing = ColumnTransformer(transformers)

    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipeline.fit(
        X_train_grouped[selected],
        y_train_grouped
    )

    validation_data = X_val_grouped[selected]

    # Warm-up prediction
    pipeline.predict(validation_data)

    # Repeat inference five times
    times = []

    for repeat in range(5):

        start = time.perf_counter()
        pipeline.predict(validation_data)
        elapsed = time.perf_counter() - start

        times.append(elapsed)

    benchmark_results.append({
        "Original Features": k,
        "Mean Prediction Time (s)": np.mean(times),
        "Std Prediction Time (s)": np.std(times),
        "Min Prediction Time (s)": np.min(times),
        "Max Prediction Time (s)": np.max(times),
        "Milliseconds per Record": (
            np.mean(times) / len(validation_data) * 1000
        )
    })

benchmark_df = pd.DataFrame(benchmark_results)

display(benchmark_df.round(6))


In [ ]:

benchmark_df.to_csv(
    "UNSW_NB15_repeated_inference_benchmark.csv",
    index=False
)

print("Repeated inference benchmark saved!")


In [ ]:

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, recall_score, confusion_matrix,
    accuracy_score
)

# Use only the official training partition.
# X_all, y_all, and groups were created in Experiment 8.

splitter = GroupShuffleSplit(
    n_splits=3,
    test_size=0.30,
    random_state=123
)

stability_results = []

for split_number, (train_idx, val_idx) in enumerate(
    splitter.split(X_all, y_all, groups),
    start=1
):
    print(f"\nStarting split {split_number}...")

    X_tr = X_all.iloc[train_idx].copy()
    X_va = X_all.iloc[val_idx].copy()
    y_tr = y_all.iloc[train_idx].copy()
    y_va = y_all.iloc[val_idx].copy()

    # Verify no identical feature groups cross the split
    assert set(groups.iloc[train_idx]).isdisjoint(
        set(groups.iloc[val_idx])
    )

    # Rank original numerical features directly.
    # Rank categorical variables as complete variables
    # using ordinal codes for MI estimation.
    ranking_data = X_tr.copy()

    discrete_columns = []

    for col in ranking_data.columns:
        if ranking_data[col].dtype == "object":
            codes, _ = pd.factorize(
                ranking_data[col].fillna("MISSING")
            )
            ranking_data[col] = codes
            discrete_columns.append(col)
        else:
            ranking_data[col] = pd.to_numeric(
                ranking_data[col], errors="coerce"
            )
            ranking_data[col] = ranking_data[col].fillna(
                ranking_data[col].median()
            )

    discrete_mask = [
        col in discrete_columns
        for col in ranking_data.columns
    ]

    mi_scores = mutual_info_classif(
        ranking_data,
        y_tr,
        discrete_features=discrete_mask,
        random_state=42
    )

    ranking = pd.Series(
        mi_scores,
        index=ranking_data.columns
    ).sort_values(ascending=False)

    ranked_features = ranking.index.tolist()

    for k in [30, 42]:

        selected = ranked_features[:k]

        numeric_selected = [
            c for c in selected
            if c not in discrete_columns
        ]

        categorical_selected = [
            c for c in selected
            if c in discrete_columns
        ]

        transformers = []

        if numeric_selected:
            transformers.append((
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_selected
            ))

        if categorical_selected:
            transformers.append((
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(
                        strategy="most_frequent"
                    )),
                    ("encoder", OneHotEncoder(
                        handle_unknown="ignore"
                    ))
                ]),
                categorical_selected
            ))

        pipeline = Pipeline([
            ("preprocessing", ColumnTransformer(
                transformers
            )),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ))
        ])

        pipeline.fit(X_tr[selected], y_tr)

        start = time.perf_counter()
        predictions = pipeline.predict(X_va[selected])
        elapsed = time.perf_counter() - start

        tn, fp, fn, tp = confusion_matrix(
            y_va, predictions, labels=[0, 1]
        ).ravel()

        stability_results.append({
            "Split": split_number,
            "Features": k,
            "Accuracy": accuracy_score(
                y_va, predictions
            ),
            "Recall": recall_score(
                y_va, predictions
            ),
            "Macro-F1": f1_score(
                y_va, predictions,
                average="macro"
            ),
            "FPR": fp / (fp + tn),
            "Missed Attacks": fn,
            "Prediction Time (s)": elapsed
        })

        print(f"Completed {k}-feature model")

stability_df = pd.DataFrame(stability_results)

print("\nAll experiment results:")
display(stability_df.round(4))


In [ ]:

stability_summary = (
    stability_df
    .groupby("Features")
    .agg({
        "Accuracy": ["mean", "std"],
        "Recall": ["mean", "std"],
        "Macro-F1": ["mean", "std"],
        "FPR": ["mean", "std"],
        "Prediction Time (s)": ["mean", "std"]
    })
)

display(stability_summary.round(4))


In [ ]:

stability_df.to_csv(
    "UNSW_NB15_three_split_stability_results.csv",
    index=False
)

stability_summary.to_csv(
    "UNSW_NB15_three_split_stability_summary.csv"
)

print("Experiment 12 results saved successfully!")


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

splitter = GroupShuffleSplit(
    n_splits=3,
    test_size=0.30,
    random_state=123
)

selected_sets = []

for split_number, (train_idx, val_idx) in enumerate(
    splitter.split(X_all, y_all, groups),
    start=1
):

    X_tr = X_all.iloc[train_idx].copy()
    y_tr = y_all.iloc[train_idx].copy()

    ranking_data = X_tr.copy()
    discrete_columns = []

    for col in ranking_data.columns:

        if ranking_data[col].dtype == "object":
            codes, _ = pd.factorize(
                ranking_data[col].fillna("MISSING")
            )
            ranking_data[col] = codes
            discrete_columns.append(col)

        else:
            ranking_data[col] = pd.to_numeric(
                ranking_data[col],
                errors="coerce"
            )
            ranking_data[col] = ranking_data[col].fillna(
                ranking_data[col].median()
            )

    discrete_mask = [
        col in discrete_columns
        for col in ranking_data.columns
    ]

    mi_scores = mutual_info_classif(
        ranking_data,
        y_tr,
        discrete_features=discrete_mask,
        random_state=42
    )

    ranking = pd.Series(
        mi_scores,
        index=ranking_data.columns
    ).sort_values(ascending=False)

    selected = set(ranking.head(30).index)
    selected_sets.append(selected)

    print(
        f"Split {split_number}:",
        len(selected),
        "features selected"
    )

# Features selected in all three splits
common_features = set.intersection(*selected_sets)

# Features selected in at least one split
all_selected = set.union(*selected_sets)

jaccard = (
    len(common_features) / len(all_selected)
)

print("\nFeatures selected in ALL splits:",
      len(common_features))

print("Features selected in ANY split:",
      len(all_selected))

print("Three-way intersection/union ratio:",
      round(jaccard, 4))

print("\nConsistently selected features:")
print(sorted(common_features))


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Use the grouped training partition from Experiment 8.
# Create an internal split so feature importance is
# measured on data not used to fit this ranking model.

inner_groups = pd.util.hash_pandas_object(
    X_train_grouped, index=False
)

inner_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026
)

inner_train_idx, inner_val_idx = next(
    inner_splitter.split(
        X_train_grouped,
        y_train_grouped,
        groups=inner_groups
    )
)

X_inner_train = X_train_grouped.iloc[inner_train_idx]
X_inner_val = X_train_grouped.iloc[inner_val_idx]

y_inner_train = y_train_grouped.iloc[inner_train_idx]
y_inner_val = y_train_grouped.iloc[inner_val_idx]

# Build preprocessing for all original variables.
numeric_inner = X_inner_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_inner = X_inner_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

importance_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"),
     numeric_inner),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent"
        )),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]), categorical_inner)
])

importance_pipeline = Pipeline([
    ("preprocessing", importance_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

importance_pipeline.fit(
    X_inner_train, y_inner_train
)

# Permutation importance operates on the original
# columns because we pass the complete pipeline.
# A smaller sample keeps the experiment manageable.
sample_size = min(5000, len(X_inner_val))

X_importance_sample = X_inner_val.sample(
    n=sample_size,
    random_state=42
)

y_importance_sample = y_inner_val.loc[
    X_importance_sample.index
]

importance = permutation_importance(
    importance_pipeline,
    X_importance_sample,
    y_importance_sample,
    scoring="f1_macro",
    n_repeats=3,
    random_state=42,
    n_jobs=1
)

permutation_ranking = pd.DataFrame({
    "Original Feature": X_inner_val.columns,
    "Importance": importance.importances_mean,
    "Std": importance.importances_std
}).sort_values(
    "Importance", ascending=False
).reset_index(drop=True)

display(permutation_ranking.head(20))


In [ ]:

permutation_ranking.to_csv(
    "UNSW_NB15_permutation_feature_ranking.csv",
    index=False
)

print("Permutation importance ranking saved!")


In [ ]:

# Compare top-30 features from both methods

mi_top30 = set(
    grouped_original_ranking["Original"].head(30)
)

permutation_top30 = set(
    permutation_ranking["Original Feature"].head(30)
)

common = mi_top30 & permutation_top30
mi_only = mi_top30 - permutation_top30
permutation_only = permutation_top30 - mi_top30

jaccard = len(common) / len(
    mi_top30 | permutation_top30
)

print("MI selected:", len(mi_top30))
print("Permutation selected:", len(permutation_top30))
print("Features selected by both:", len(common))
print("Jaccard similarity:", round(jaccard, 4))

print("\nCommon features:")
print(sorted(common))

print("\nOnly selected by MI:")
print(sorted(mi_only))

print("\nOnly selected by permutation importance:")
print(sorted(permutation_only))


In [ ]:

import pandas as pd
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

mi_features = (
    grouped_original_ranking["Original"]
    .head(30).tolist()
)

permutation_features = (
    permutation_ranking["Original Feature"]
    .head(30).tolist()
)

all_features = X_train_grouped.columns.tolist()

feature_sets = {
    "MI - 30 features": mi_features,
    "Permutation - 30 features": permutation_features,
    "All - 42 features": all_features
}

experiment15_results = []

for name, selected in feature_sets.items():

    print("Training:", name)

    numeric_selected = (
        X_train_grouped[selected]
        .select_dtypes(include="number")
        .columns.tolist()
    )

    categorical_selected = (
        X_train_grouped[selected]
        .select_dtypes(include=["object", "category"])
        .columns.tolist()
    )

    transformers = []

    if numeric_selected:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_selected
        ))

    if categorical_selected:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_selected
        ))

    model_pipeline = Pipeline([
        ("preprocessing", ColumnTransformer(
            transformers
        )),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    start = time.perf_counter()

    model_pipeline.fit(
        X_train_grouped[selected],
        y_train_grouped
    )

    train_time = time.perf_counter() - start

    start = time.perf_counter()

    predictions = model_pipeline.predict(
        X_val_grouped[selected]
    )

    prediction_time = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(
        y_val_grouped,
        predictions,
        labels=[0, 1]
    ).ravel()

    experiment15_results.append({
        "Method": name,
        "Features": len(selected),
        "Accuracy": accuracy_score(
            y_val_grouped, predictions
        ),
        "Precision": precision_score(
            y_val_grouped, predictions
        ),
        "Recall": recall_score(
            y_val_grouped, predictions
        ),
        "Macro-F1": f1_score(
            y_val_grouped, predictions,
            average="macro"
        ),
        "FPR": fp / (fp + tn),
        "Missed Attacks": int(fn),
        "Training Time (s)": train_time,
        "Prediction Time (s)": prediction_time
    })

results15_df = pd.DataFrame(experiment15_results)

display(results15_df.round(4))


In [ ]:

results15_df.to_csv(
    "UNSW_NB15_MI_vs_Permutation_vs_All.csv",
    index=False
)

print("Experiment 15 results saved successfully!")


In [ ]:

import pandas as pd

# X_all contains the 42 original network features.
# y_all contains the binary attack label.

conflict_check = pd.DataFrame({
    "Group": groups.to_numpy(),
    "Label": y_all.to_numpy()
})

group_statistics = conflict_check.groupby("Group").agg(
    Records=("Label", "size"),
    Distinct_Labels=("Label", "nunique")
)

conflicting_groups = group_statistics[
    group_statistics["Distinct_Labels"] > 1
]

print(
    "Conflicting feature-vector groups:",
    len(conflicting_groups)
)

print(
    "Records in conflicting groups:",
    int(conflicting_groups["Records"].sum())
)

print(
    "Percentage of training records affected:",
    round(
        conflicting_groups["Records"].sum()
        / len(X_all) * 100,
        4
    ),
    "%"
)

print(
    "Largest conflicting group size:",
    int(conflicting_groups["Records"].max())
)

display(
    conflicting_groups.sort_values(
        "Records", ascending=False
    ).head(10)
)


In [ ]:

import pandas as pd

# Identify groups that have both normal and attack labels.
conflicting_group_ids = set(
    conflicting_groups.index
)

# Reconstruct group IDs for the existing
# duplicate-aware training and validation partitions.
train_group_ids = pd.util.hash_pandas_object(
    X_train_grouped,
    index=False
)

val_group_ids = pd.util.hash_pandas_object(
    X_val_grouped,
    index=False
)

train_conflict_mask = train_group_ids.isin(
    conflicting_group_ids
)

val_conflict_mask = val_group_ids.isin(
    conflicting_group_ids
)

print("TRAINING PARTITION")
print("Total records:", len(X_train_grouped))
print(
    "Records in conflicting groups:",
    int(train_conflict_mask.sum())
)
print(
    "Percentage:",
    round(train_conflict_mask.mean() * 100, 4),
    "%"
)

print("\nVALIDATION PARTITION")
print("Total records:", len(X_val_grouped))
print(
    "Records in conflicting groups:",
    int(val_conflict_mask.sum())
)
print(
    "Percentage:",
    round(val_conflict_mask.mean() * 100, 4),
    "%"
)

print("\nGROUP OVERLAP CHECK")
print(
    "Conflicting groups appearing in both partitions:",
    len(
        set(train_group_ids[train_conflict_mask])
        & set(val_group_ids[val_conflict_mask])
    )
)


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Experiment 15 ended with the 42-feature model.
# model_pipeline therefore refers to that model,
# provided the notebook cells have not overwritten it.

assert len(all_features) == 42

validation_predictions = model_pipeline.predict(
    X_val_grouped[all_features]
)

# Use the mask from Experiment 16.
conflict_mask = val_conflict_mask.to_numpy()

def evaluate_subset(name, mask):
    actual = y_val_grouped.to_numpy()[mask]
    predicted = validation_predictions[mask]

    tn, fp, fn, tp = confusion_matrix(
        actual, predicted, labels=[0, 1]
    ).ravel()

    return {
        "Subset": name,
        "Records": len(actual),
        "Normal": int((actual == 0).sum()),
        "Attacks": int((actual == 1).sum()),
        "Errors": int((actual != predicted).sum()),
        "Accuracy": accuracy_score(actual, predicted),
        "Macro-F1": f1_score(
            actual, predicted,
            average="macro",
            zero_division=0
        ),
        "Attack Recall": (
            tp / (tp + fn) if tp + fn else float("nan")
        ),
        "FPR": (
            fp / (fp + tn) if fp + tn else float("nan")
        ),
        "Missed Attacks": int(fn)
    }

conflict_performance = pd.DataFrame([
    evaluate_subset(
        "Conflicting groups",
        conflict_mask
    ),
    evaluate_subset(
        "Non-conflicting groups",
        ~conflict_mask
    ),
    evaluate_subset(
        "All validation records",
        [True] * len(y_val_grouped)
    )
])

display(conflict_performance.round(4))


In [ ]:

conflict_performance.to_csv(
    "UNSW_NB15_conflicting_label_performance.csv",
    index=False
)

print("Experiment 17 results saved!")


In [ ]:

import pandas as pd

# Examine the original training file without modifying it.

group_details = pd.DataFrame({
    "Group": groups.to_numpy(),
    "Label": y_all.to_numpy()
})

label_counts = (
    group_details
    .groupby(["Group", "Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
)

label_counts.columns = [
    "Normal Records",
    "Attack Records"
]

label_counts["Total Records"] = (
    label_counts["Normal Records"]
    + label_counts["Attack Records"]
)

# Keep groups containing both labels.
conflict_details = label_counts[
    (label_counts["Normal Records"] > 0)
    & (label_counts["Attack Records"] > 0)
].copy()

conflict_details["Attack Proportion"] = (
    conflict_details["Attack Records"]
    / conflict_details["Total Records"]
)

print("Number of conflicting groups:",
      len(conflict_details))

print("\nTotal normal records in conflicting groups:",
      conflict_details["Normal Records"].sum())

print("\nTotal attack records in conflicting groups:",
      conflict_details["Attack Records"].sum())

print("\nGroups with more attack than normal records:",
      (
          conflict_details["Attack Records"]
          > conflict_details["Normal Records"]
      ).sum())

print("\nGroups with more normal than attack records:",
      (
          conflict_details["Normal Records"]
          > conflict_details["Attack Records"]
      ).sum())

print("\nGroups with equal numbers of both labels:",
      (
          conflict_details["Normal Records"]
          == conflict_details["Attack Records"]
      ).sum())

display(
    conflict_details.sort_values(
        "Total Records",
        ascending=False
    ).head(15)
)


In [ ]:

# Theoretical minimum errors when identical
# feature vectors must receive identical predictions.

conflict_details["Minimum Unavoidable Errors"] = (
    conflict_details[
        ["Normal Records", "Attack Records"]
    ].min(axis=1)
)

minimum_errors = int(
    conflict_details[
        "Minimum Unavoidable Errors"
    ].sum()
)

total_conflicting_records = int(
    conflict_details["Total Records"].sum()
)

print("Conflicting groups:", len(conflict_details))

print(
    "Records in conflicting groups:",
    total_conflicting_records
)

print(
    "Minimum unavoidable errors:",
    minimum_errors
)

print(
    "Minimum error rate within conflicting groups:",
    round(
        minimum_errors / total_conflicting_records * 100,
        2
    ),
    "%"
)

print(
    "Minimum error rate across full training dataset:",
    round(
        minimum_errors / len(X_all) * 100,
        4
    ),
    "%"
)


In [ ]:

import json
from datetime import datetime, timezone

research_protocol = {
    "experiment": "Experiment 20",
    "dataset": "UNSW-NB15",
    "task": "Binary intrusion detection",
    "target": "label",
    "input_features": 42,

    "data_policy": {
        "official_training_rows": 175341,
        "official_test_rows": 82332,
        "official_test_usage": "Final evaluation only",
        "remove_duplicate_records": False,
        "remove_conflicting_labels": False,
        "exclude_columns": ["id", "attack_cat", "label"]
    },

    "validation": {
        "method": "GroupShuffleSplit",
        "group_definition": "Identical original feature vectors",
        "validation_fraction": 0.30,
        "random_seeds": [42, 43, 44],
        "feature_selection_within_each_split": True
    },

    "models": {
        "Logistic Regression": {
            "max_iter": 1000
        },
        "Random Forest": {
            "n_estimators": 100,
            "max_depth": 20,
            "min_samples_leaf": 2
        }
    },

    "feature_selection": {
        "methods": [
            "Mutual Information",
            "Permutation Importance",
            "All original features"
        ],
        "selected_feature_count": 30
    },

    "primary_metric": "Macro-F1",
    "secondary_metrics": [
        "Attack recall",
        "False-positive rate",
        "Precision",
        "Training time",
        "Prediction time"
    ],

    "status": "Protocol draft - not yet finalized"
}

with open(
    "UNSW_NB15_research_protocol.json",
    "w"
) as file:
    json.dump(research_protocol, file, indent=4)

print("Research protocol created successfully!")
print()
print(json.dumps(research_protocol, indent=4))


In [ ]:

print("Training dataset:", train_df.shape)
print("Official test dataset:", test_df.shape)

print("Grouped training features:", X_train_grouped.shape)
print("Grouped validation features:", X_val_grouped.shape)

print("Grouped training labels:", len(y_train_grouped))
print("Grouped validation labels:", len(y_val_grouped))

assert train_df.shape[0] == 175341
assert test_df.shape[0] == 82332

assert X_train_grouped.shape[1] == 42
assert X_val_grouped.shape[1] == 42

print("\nAll basic checks passed!")


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Use the official training data only.
X_all = train_df.drop(
    columns=["id", "attack_cat", "label"]
).copy()

y_all = train_df["label"].copy()

# Give identical original feature vectors the same group.
groups = pd.util.hash_pandas_object(
    X_all,
    index=False
)

def rank_original_features_mi(X_train, y_train):

    numeric_cols = X_train.select_dtypes(
        include="number"
    ).columns.tolist()

    categorical_cols = X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    # Impute numerical values.
    numeric_data = SimpleImputer(
        strategy="median"
    ).fit_transform(X_train[numeric_cols])

    # Impute and ordinal-encode categorical values.
    if categorical_cols:
        categorical_imputer = SimpleImputer(
            strategy="most_frequent"
        )

        categorical_data = categorical_imputer.fit_transform(
            X_train[categorical_cols]
        )

        categorical_data = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ).fit_transform(categorical_data)

        processed = np.column_stack([
            numeric_data,
            categorical_data
        ])
    else:
        processed = numeric_data

    feature_names = numeric_cols + categorical_cols

    # Mark categorical columns as discrete.
    discrete_mask = (
        [False] * len(numeric_cols)
        + [True] * len(categorical_cols)
    )

    scores = mutual_info_classif(
        processed,
        y_train,
        discrete_features=discrete_mask,
        random_state=42
    )

    ranking = pd.DataFrame({
        "Feature": feature_names,
        "MI": scores
    }).sort_values(
        "MI",
        ascending=False
    ).reset_index(drop=True)

    return ranking


# Start with seed 42 as a functionality check.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        X_all,
        y_all,
        groups=groups
    )
)

X_train_21 = X_all.iloc[train_idx].copy()
X_val_21 = X_all.iloc[val_idx].copy()

y_train_21 = y_all.iloc[train_idx].copy()
y_val_21 = y_all.iloc[val_idx].copy()

ranking_21 = rank_original_features_mi(
    X_train_21,
    y_train_21
)

print("Training records:", len(X_train_21))
print("Validation records:", len(X_val_21))
print("Features ranked:", len(ranking_21))

display(ranking_21.head(15))


In [ ]:

# EXPERIMENT 21 - STEP 2
# Compare MI-selected 30 features against all 42 features

import pandas as pd
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Select the top 30 features from Step 1
mi_30_features = ranking_21["Feature"].head(30).tolist()

# 2. Define the two configurations
feature_sets_21 = {
    "MI - 30 original features": mi_30_features,
    "All - 42 original features": X_train_21.columns.tolist()
}

results_21 = []

# 3. Train and evaluate each model
for method_name, selected_features in feature_sets_21.items():

    print("Training:", method_name)

    # Identify numerical and categorical columns
    numeric_cols = (
        X_train_21[selected_features]
        .select_dtypes(include="number")
        .columns.tolist()
    )

    categorical_cols = (
        X_train_21[selected_features]
        .select_dtypes(include=["object", "category"])
        .columns.tolist()
    )

    transformers = []

    # Prepare numerical features
    if numeric_cols:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_cols
        ))

    # Prepare categorical features
    if categorical_cols:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_cols
        ))

    # Build complete machine-learning pipeline
    pipeline = Pipeline([
        (
            "preprocessing",
            ColumnTransformer(transformers)
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    # Measure training time
    start = time.perf_counter()

    pipeline.fit(
        X_train_21[selected_features],
        y_train_21
    )

    training_time = time.perf_counter() - start

    # Measure prediction time
    start = time.perf_counter()

    predictions = pipeline.predict(
        X_val_21[selected_features]
    )

    prediction_time = time.perf_counter() - start

    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_val_21,
        predictions,
        labels=[0, 1]
    ).ravel()

    # Save results
    results_21.append({
        "Method": method_name,
        "Original Features": len(selected_features),

        "Accuracy": accuracy_score(
            y_val_21, predictions
        ),

        "Precision": precision_score(
            y_val_21, predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_val_21, predictions,
            zero_division=0
        ),

        "Macro-F1": f1_score(
            y_val_21,
            predictions,
            average="macro"
        ),

        "FPR": fp / (fp + tn),

        "Missed Attacks": int(fn),

        "Training Time (s)": training_time,

        "Prediction Time (s)": prediction_time
    })

# 4. Display comparison table
results_21_df = pd.DataFrame(results_21)

print("\nEXPERIMENT 21 - COMPARISON RESULTS")

display(results_21_df.round(4))


In [ ]:

results_21_df.to_csv(
    "UNSW_NB15_Experiment21_MI30_vs_All42.csv",
    index=False
)

print("Experiment 21 results saved successfully!")


In [ ]:

# EXPERIMENT 22
# Repeated duplicate-aware comparison:
# MI-30 vs All-42

import pandas as pd
import time

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix
)

experiment22_results = []

for seed in [42, 43, 44]:

    print(f"\nStarting split {seed}...")

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=seed
    )

    train_idx, val_idx = next(
        splitter.split(X_all, y_all, groups=groups)
    )

    X_tr = X_all.iloc[train_idx].copy()
    X_va = X_all.iloc[val_idx].copy()
    y_tr = y_all.iloc[train_idx].copy()
    y_va = y_all.iloc[val_idx].copy()

    # Rank features using training data only.
    mi_start = time.perf_counter()

    ranking = rank_original_features_mi(X_tr, y_tr)

    mi_time = time.perf_counter() - mi_start

    feature_sets = {
        "MI-30": ranking["Feature"].head(30).tolist(),
        "All-42": X_tr.columns.tolist()
    }

    for method, selected in feature_sets.items():

        numeric_cols = X_tr[selected].select_dtypes(
            include="number"
        ).columns.tolist()

        categorical_cols = X_tr[selected].select_dtypes(
            include=["object", "category"]
        ).columns.tolist()

        transformers = []

        if numeric_cols:
            transformers.append((
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_cols
            ))

        if categorical_cols:
            transformers.append((
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(
                        strategy="most_frequent"
                    )),
                    ("encoder", OneHotEncoder(
                        handle_unknown="ignore"
                    ))
                ]),
                categorical_cols
            ))

        model = Pipeline([
            ("preprocessing", ColumnTransformer(
                transformers
            )),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ))
        ])

        start = time.perf_counter()

        model.fit(X_tr[selected], y_tr)

        train_time = time.perf_counter() - start

        start = time.perf_counter()

        predictions = model.predict(X_va[selected])

        predict_time = time.perf_counter() - start

        tn, fp, fn, tp = confusion_matrix(
            y_va, predictions, labels=[0, 1]
        ).ravel()

        experiment22_results.append({
            "Seed": seed,
            "Method": method,
            "Features": len(selected),
            "Accuracy": accuracy_score(y_va, predictions),
            "Recall": recall_score(y_va, predictions),
            "Macro-F1": f1_score(
                y_va, predictions, average="macro"
            ),
            "FPR": fp / (fp + tn),
            "Missed Attacks": int(fn),
            "MI Ranking Time (s)": (
                mi_time if method == "MI-30" else 0
            ),
            "Training Time (s)": train_time,
            "Prediction Time (s)": predict_time
        })

        print(f"Completed: {method}")

results22_df = pd.DataFrame(experiment22_results)

print("\nALL EXPERIMENT 22 RESULTS")
display(results22_df.round(4))

print("\nAVERAGE RESULTS")
summary22 = (
    results22_df
    .groupby("Method")
    .agg({
        "Accuracy": ["mean", "std"],
        "Recall": ["mean", "std"],
        "Macro-F1": ["mean", "std"],
        "FPR": ["mean", "std"],
        "Training Time (s)": ["mean", "std"],
        "Prediction Time (s)": ["mean", "std"]
    })
)

display(summary22.round(4))


In [ ]:

results22_df.to_csv(
    "UNSW_NB15_Experiment22_repeated_MI_comparison.csv",
    index=False
)

summary22.to_csv(
    "UNSW_NB15_Experiment22_summary.csv"
)

print("Experiment 22 results saved!")


In [ ]:

# EXPERIMENT 23
# Check MI-30 feature stability across three splits

from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

selected_sets_23 = {}

for seed in [42, 43, 44]:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=seed
    )

    train_idx, val_idx = next(
        splitter.split(X_all, y_all, groups=groups)
    )

    X_tr = X_all.iloc[train_idx].copy()
    y_tr = y_all.iloc[train_idx].copy()

    # Recalculate ranking on training records only
    ranking = rank_original_features_mi(X_tr, y_tr)

    selected = set(
        ranking["Feature"].head(30).tolist()
    )

    selected_sets_23[seed] = selected

    print(
        f"Split {seed}: {len(selected)} features selected"
    )

# Features common to all three splits
common_features = set.intersection(
    *selected_sets_23.values()
)

# Features selected in at least one split
union_features = set.union(
    *selected_sets_23.values()
)

stability_ratio = (
    len(common_features) / len(union_features)
)

print("\nSTABILITY RESULTS")
print("Common features:", len(common_features))
print("Union of features:", len(union_features))
print("Intersection/union ratio:", round(stability_ratio, 4))

print("\nFeatures selected in all three splits:")
print(sorted(common_features))


In [ ]:

# EXPERIMENT 24
# Save feature-selection stability results

import pandas as pd

# Record whether each feature was selected in each split.
all_features = sorted(union_features)

stability_table = pd.DataFrame({
    "Feature": all_features,
    "Selected_Seed_42": [
        int(f in selected_sets_23[42]) for f in all_features
    ],
    "Selected_Seed_43": [
        int(f in selected_sets_23[43]) for f in all_features
    ],
    "Selected_Seed_44": [
        int(f in selected_sets_23[44]) for f in all_features
    ]
})

# Count how often each feature was selected.
stability_table["Selection_Frequency"] = (
    stability_table[
        ["Selected_Seed_42",
         "Selected_Seed_43",
         "Selected_Seed_44"]
    ].sum(axis=1)
)

stability_table = stability_table.sort_values(
    ["Selection_Frequency", "Feature"],
    ascending=[False, True]
).reset_index(drop=True)

display(stability_table)

stability_table.to_csv(
    "UNSW_NB15_Experiment24_feature_stability.csv",
    index=False
)

print("\nFeatures selected in all three splits:",
      (stability_table["Selection_Frequency"] == 3).sum())

print("Features selected in exactly two splits:",
      (stability_table["Selection_Frequency"] == 2).sum())

print("Features selected in exactly one split:",
      (stability_table["Selection_Frequency"] == 1).sum())

print("\nExperiment 24 results saved successfully!")


In [ ]:

# EXPERIMENT 25
# Audit exact feature-vector overlap between
# official training and official test datasets.

import pandas as pd

# Use the same 42 input features in both datasets.
feature_columns = X_all.columns.tolist()

train_features = train_df[feature_columns].copy()
test_features = test_df[feature_columns].copy()

# Create hashes from the original feature values.
train_hashes = pd.util.hash_pandas_object(
    train_features,
    index=False
)

test_hashes = pd.util.hash_pandas_object(
    test_features,
    index=False
)

# Check which test records have identical
# feature vectors somewhere in official training.
test_overlap_mask = test_hashes.isin(
    set(train_hashes)
)

overlap_count = int(test_overlap_mask.sum())

print("OFFICIAL TRAIN / TEST OVERLAP AUDIT")
print("Official training records:", len(train_features))
print("Official test records:", len(test_features))

print("\nTest records with features also in training:",
      overlap_count)

print("Percentage of official test records:",
      round(overlap_count / len(test_features) * 100, 4),
      "%")

print("\nTest records without exact feature overlap:",
      len(test_features) - overlap_count)


In [ ]:

# EXPERIMENT 26
# Audit training-label consistency for feature
# patterns appearing in the official test set.

import pandas as pd

# Group official training records by their feature hashes.
training_label_audit = pd.DataFrame({
    "Feature_Hash": train_hashes.to_numpy(),
    "Label": train_df["label"].to_numpy()
})

training_group_summary = (
    training_label_audit
    .groupby("Feature_Hash")["Label"]
    .agg(["size", "nunique"])
)

# Identify feature patterns with conflicting training labels.
conflicting_training_hashes = set(
    training_group_summary.index[
        training_group_summary["nunique"] > 1
    ]
)

# Identify official test records whose feature patterns
# appear in those conflicting training groups.
test_conflict_mask = test_hashes.isin(
    conflicting_training_hashes
)

print("EXPERIMENT 26 - OVERLAP LABEL AUDIT")

print(
    "Official test records with training feature overlap:",
    int(test_overlap_mask.sum())
)

print(
    "Overlapping test records matching conflicting "
    "training-label groups:",
    int(test_conflict_mask.sum())
)

print(
    "Overlapping test records matching consistent "
    "training-label groups:",
    int((test_overlap_mask & ~test_conflict_mask).sum())
)

print(
    "\nPercentage of overlapping test records matching "
    "conflicting training-label groups:",
    round(
        test_conflict_mask.sum()
        / test_overlap_mask.sum() * 100,
        2
    ),
    "%"
)


In [ ]:

# EXPERIMENT 27
# FINAL OFFICIAL TEST EVALUATION
# MI-30 versus All-42
# Run once, after freezing the protocol.

import pandas as pd
import numpy as np
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Use the complete official training dataset.
#    Exclude ID and both label columns from the inputs.

feature_columns = X_all.columns.tolist()

X_final_train = train_df[feature_columns].copy()
y_final_train = train_df["label"].copy()

X_final_test = test_df[feature_columns].copy()
y_final_test = test_df["label"].copy()

assert len(X_final_train) == 175341
assert len(X_final_test) == 82332
assert len(feature_columns) == 42

print("Official training rows:", len(X_final_train))
print("Official test rows:", len(X_final_test))

# 2. Recalculate MI ranking using ONLY official training.
print("\nRanking features using training data only...")

mi_start = time.perf_counter()

final_mi_ranking = rank_original_features_mi(
    X_final_train,
    y_final_train
)

mi_ranking_seconds = time.perf_counter() - mi_start

final_mi_30 = final_mi_ranking["Feature"].head(30).tolist()

print("\nFinal MI-30 features:")
print(final_mi_30)

# 3. Freeze the two feature configurations.
final_feature_sets = {
    "MI-30": final_mi_30,
    "All-42": feature_columns
}

def make_final_pipeline(selected_features):

    numeric_cols = (
        X_final_train[selected_features]
        .select_dtypes(include="number")
        .columns.tolist()
    )

    categorical_cols = (
        X_final_train[selected_features]
        .select_dtypes(include=["object", "category"])
        .columns.tolist()
    )

    transformers = []

    if numeric_cols:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_cols
        ))

    if categorical_cols:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_cols
        ))

    return Pipeline([
        ("preprocessing", ColumnTransformer(transformers)),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

# 4. Prepare overlap masks from Experiment 25.
#    These use input features only, not test labels.

train_hashes_final = pd.util.hash_pandas_object(
    X_final_train,
    index=False
)

test_hashes_final = pd.util.hash_pandas_object(
    X_final_test,
    index=False
)

overlap_mask_final = test_hashes_final.isin(
    set(train_hashes_final)
).to_numpy()

# 5. Metric function.
def evaluate_final_subset(y_true, y_pred, method, subset):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "Method": method,
        "Subset": subset,
        "Records": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "Macro-F1": f1_score(
            y_true, y_pred, average="macro"
        ),
        "FPR": (
            fp / (fp + tn) if (fp + tn) > 0 else np.nan
        ),
        "Missed Attacks": int(fn)
    }

final_results = []
final_models = {}
final_predictions = {}
final_timings = []

# 6. Train both frozen models and evaluate once.
for method, selected_features in final_feature_sets.items():

    print("\nTraining final model:", method)

    pipeline = make_final_pipeline(selected_features)

    start = time.perf_counter()

    pipeline.fit(
        X_final_train[selected_features],
        y_final_train
    )

    train_seconds = time.perf_counter() - start

    start = time.perf_counter()

    predictions = pipeline.predict(
        X_final_test[selected_features]
    )

    prediction_seconds = time.perf_counter() - start

    final_models[method] = pipeline
    final_predictions[method] = predictions

    # Full official test set
    final_results.append(
        evaluate_final_subset(
            y_final_test,
            predictions,
            method,
            "All official test records"
        )
    )

    # Exact feature overlap with training
    final_results.append(
        evaluate_final_subset(
            y_final_test.iloc[
                np.flatnonzero(overlap_mask_final)
            ],
            predictions[overlap_mask_final],
            method,
            "Overlapping test records"
        )
    )

    # No exact feature overlap with training
    final_results.append(
        evaluate_final_subset(
            y_final_test.iloc[
                np.flatnonzero(~overlap_mask_final)
            ],
            predictions[~overlap_mask_final],
            method,
            "Non-overlapping test records"
        )
    )

    final_timings.append({
        "Method": method,
        "Features": len(selected_features),
        "MI Ranking Time (s)": (
            mi_ranking_seconds if method == "MI-30" else 0
        ),
        "Training Time (s)": train_seconds,
        "Prediction Time (s)": prediction_seconds
    })

    print("Completed:", method)

# 7. Display results.
final_results_df = pd.DataFrame(final_results)
final_timings_df = pd.DataFrame(final_timings)

print("\nFINAL TEST PERFORMANCE")
display(final_results_df.round(4))

print("\nFINAL COMPUTATIONAL COST")
display(final_timings_df.round(4))

# 8. Save research results.
final_results_df.to_csv(
    "UNSW_NB15_Experiment27_final_test_results.csv",
    index=False
)

final_timings_df.to_csv(
    "UNSW_NB15_Experiment27_final_timings.csv",
    index=False
)

final_mi_ranking.to_csv(
    "UNSW_NB15_Experiment27_final_MI_ranking.csv",
    index=False
)

print("\nFinal results saved successfully!")


In [ ]:

# EXPERIMENT 28 — STEP 1
# Consolidate and audit final research results.
# No retraining. No test-set tuning.

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Check that Experiment 27 results are available.
required_objects = [
    "final_results_df",
    "final_timings_df",
    "final_mi_ranking",
    "final_mi_30",
    "X_final_train",
    "X_final_test",
    "y_final_train",
    "y_final_test",
    "overlap_mask_final"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    print("Missing notebook variables:", missing)
    print("Please restore the saved Experiment 27 results")
    print("or rerun the notebook's setup cells.")
else:
    # Check dataset sizes and feature selection.
    assert len(X_final_train) == 175341
    assert len(X_final_test) == 82332
    assert X_final_train.shape[1] == 42
    assert X_final_test.shape[1] == 42
    assert len(final_mi_30) == 30
    assert len(set(final_mi_30)) == 30

    # Check that both models were evaluated on
    # all three official test subsets.
    assert len(final_results_df) == 6
    assert len(final_timings_df) == 2

    expected_methods = {"MI-30", "All-42"}
    assert set(final_results_df["Method"]) == expected_methods

    # Check overlap and non-overlap counts.
    overlap_count = int(np.sum(overlap_mask_final))
    nonoverlap_count = len(X_final_test) - overlap_count

    assert overlap_count == 8541
    assert nonoverlap_count == 73791

    # Confirm subset counts sum to official test size.
    for method in expected_methods:
        subset = final_results_df[
            final_results_df["Method"] == method
        ]

        full = subset[
            subset["Subset"] == "All official test records"
        ]["Records"].iloc[0]

        overlap = subset[
            subset["Subset"] == "Overlapping test records"
        ]["Records"].iloc[0]

        nonoverlap = subset[
            subset["Subset"] == "Non-overlapping test records"
        ]["Records"].iloc[0]

        assert full == overlap + nonoverlap == 82332

    # Create the main publication-ready comparison.
    final_summary = final_results_df[
        final_results_df["Subset"] ==
        "All official test records"
    ].copy()

    final_summary["Feature Count"] = (
        final_summary["Method"].map({
            "MI-30": 30,
            "All-42": 42
        })
    )

    final_summary = final_summary[
        [
            "Method",
            "Feature Count",
            "Accuracy",
            "Precision",
            "Recall",
            "Macro-F1",
            "FPR",
            "Missed Attacks"
        ]
    ]

    # Summarize reproducibility information.
    audit = {
        "Experiment": 28,
        "Purpose": "Final results consolidation",
        "Dataset": "UNSW-NB15",
        "Official training records": len(X_final_train),
        "Official test records": len(X_final_test),
        "Original input features": 42,
        "MI selected features": len(final_mi_30),
        "Feature reduction percent": round(
            (42 - 30) / 42 * 100, 2
        ),
        "Overlapping test records": overlap_count,
        "Non-overlapping test records": nonoverlap_count,
        "Models": ["MI-30", "All-42"],
        "Classifier": "Random Forest",
        "Trees": 100,
        "Maximum depth": 20,
        "Minimum samples per leaf": 2,
        "Random state": 42,
        "Primary metric": "Macro-F1",
        "Official test used for tuning": False
    }

    # Save outputs.
    output_folder = Path("experiment28")
    output_folder.mkdir(exist_ok=True)

    final_summary.to_csv(
        output_folder / "final_performance_summary.csv",
        index=False
    )

    final_results_df.to_csv(
        output_folder / "final_subset_results.csv",
        index=False
    )

    final_timings_df.to_csv(
        output_folder / "final_computational_cost.csv",
        index=False
    )

    final_mi_ranking.to_csv(
        output_folder / "final_feature_ranking.csv",
        index=False
    )

    with open(
        output_folder / "research_audit.json",
        "w"
    ) as file:
        json.dump(audit, file, indent=4)

    print("EXPERIMENT 28 — AUDIT PASSED")
    print("\nFINAL MODEL COMPARISON")
    display(final_summary.round(4))

    print("\nRESEARCH AUDIT")
    for key, value in audit.items():
        print(f"{key}: {value}")

    print("\nFiles saved in:", output_folder)


In [ ]:

# EXPERIMENT 28 — STEP 2
# Record software versions, dataset fingerprints,
# selected features and trained model settings.
# This cell does NOT retrain or evaluate models.

import sys
import json
import hashlib
import platform
from pathlib import Path
from importlib.metadata import version

import pandas as pd
import numpy as np
import sklearn

output_folder = Path("experiment28")
output_folder.mkdir(exist_ok=True)

# Create a fingerprint of the dataset contents.
# The fingerprint helps identify which exact
# DataFrame content was used.
def dataframe_fingerprint(df):
    row_hashes = pd.util.hash_pandas_object(
        df, index=False
    ).to_numpy(dtype="uint64")

    digest = hashlib.sha256()
    digest.update(
        json.dumps(df.columns.tolist()).encode("utf-8")
    )
    digest.update(
        json.dumps(
            df.dtypes.astype(str).tolist()
        ).encode("utf-8")
    )
    digest.update(row_hashes.tobytes())

    return digest.hexdigest()

# Record trained Random Forest settings.
model_settings = {}

for method, pipeline in final_models.items():
    classifier = pipeline.named_steps["classifier"]

    model_settings[method] = {
        "class": type(classifier).__name__,
        "n_estimators": classifier.n_estimators,
        "max_depth": classifier.max_depth,
        "min_samples_leaf": classifier.min_samples_leaf,
        "random_state": classifier.random_state,
        "n_jobs": classifier.n_jobs
    }

reproducibility_record = {
    "experiment": "Experiment 28",
    "dataset": "UNSW-NB15",
    "python_version": sys.version.split()[0],
    "operating_system": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "training_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "training_fingerprint": dataframe_fingerprint(train_df),
    "test_fingerprint": dataframe_fingerprint(test_df),
    "input_feature_names": feature_columns,
    "selected_MI30_features": final_mi_30,
    "feature_ranking": "Mutual information",
    "feature_ranking_source": "Official training data only",
    "models": model_settings,
    "primary_metric": "Macro-F1",
    "test_evaluation_status": "Completed; no further tuning"
}

record_path = (
    output_folder / "reproducibility_record.json"
)

with open(record_path, "w", encoding="utf-8") as f:
    json.dump(
        reproducibility_record,
        f,
        indent=4
    )

print("REPRODUCIBILITY RECORD CREATED")
print("Python:", reproducibility_record["python_version"])
print("Pandas:", reproducibility_record["pandas_version"])
print("NumPy:", reproducibility_record["numpy_version"])
print(
    "Scikit-learn:",
    reproducibility_record["scikit_learn_version"]
)

print("\nDataset fingerprints:")
print(
    "Training:",
    reproducibility_record["training_fingerprint"][:20],
    "..."
)
print(
    "Test:",
    reproducibility_record["test_fingerprint"][:20],
    "..."
)

print("\nSelected MI-30 features:")
print(final_mi_30)

print("\nModel settings:")
print(json.dumps(model_settings, indent=4))

print("\nSaved to:", record_path)


In [ ]:

# EXPERIMENT 28 — STEP 3
# Package research evidence for safekeeping.
# No model training or tuning.

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

folder = Path("experiment28")

required_files = [
    "final_performance_summary.csv",
    "final_subset_results.csv",
    "final_computational_cost.csv",
    "final_feature_ranking.csv",
    "research_audit.json",
    "reproducibility_record.json"
]

# Check that every required file exists.
missing_files = [
    name for name in required_files
    if not (folder / name).exists()
]

if missing_files:
    print("Missing files:")
    print(missing_files)
    print("Please rerun Experiment 28 Steps 1 and 2.")
else:
    zip_path = Path(
        "UNSW_NB15_Experiment28_Research_Evidence.zip"
    )

    with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
        for filename in required_files:
            archive.write(
                folder / filename,
                arcname=filename
            )

    print("RESEARCH EVIDENCE PACKAGE CREATED")
    print("Files included:", len(required_files))
    print("ZIP file:", zip_path)
    print(
        "Size:",
        round(zip_path.stat().st_size / 1024, 2),
        "KB"
    )

    files.download(str(zip_path))
